In [1]:
import importlib

In [2]:


import pandas as pd
import data_loader
import user_profile as user_pf


In [3]:
importlib.reload(user_pf)

<module 'user_profile' from '/Users/Cathal/Rec-Genie/rec-sys/user_profile.py'>

In [4]:
# Change this to True if you are running on Google Colab
RUNNING_ON_COLAB = False

# Constants
USER_ID = 999999

In [5]:
if RUNNING_ON_COLAB:
    data_loader.mount_drive()

In [6]:

# films_df = data_loader.load_movies()

In [7]:
# Load the data
films_df = data_loader.load_movies()

In [8]:
load_original_credits = False
credits_df = data_loader.load_credits(load_original=load_original_credits)

In [9]:
ratings_df = data_loader.load_ratings()


In [10]:
print("Data dimensions:")
print("films_df: ", films_df.shape)
print("ratings_df: ", ratings_df.shape)
print("credits_df: ", credits_df.shape)

Data dimensions:
films_df:  (1070323, 8)
ratings_df:  (32000204, 3)
credits_df:  (45476, 3)


In [11]:
credits_df.shape

(45476, 3)

In [12]:
films_df.columns

Index(['id', 'title', 'vote_average', 'vote_count', 'release_date', 'imdb_id',
       'popularity', 'genres'],
      dtype='object')

In [13]:
# Find film_df entry 100 - Lock Stock and Two Smoking Barrels
films_df[films_df['id'] == 100]

,id,title,vote_average,vote_count,release_date,imdb_id,popularity,genres
65,100,"Lock, Stock and Two Smoking Barrels",8.115,6692.0,1998-08-28,tt0120735,1.946,"Comedy, Crime"


In [14]:
# Drop the weird film entry
films_df = films_df.drop(35587)

In [15]:
import preprocessing as pre
# importlib.reload(pre)

In [16]:
# Drop films before 1985 and after today
films_df = pre.filter_films(films_df)


In [17]:
films_df.shape

(92114, 8)

In [18]:
films_df.sort_values('release_date', ascending=False)

,id,title,vote_average,vote_count,release_date,imdb_id,popularity,genres
304132,447273,Snow White,2.5,22.0,2025-03-19,tt6208148,24.058,"Family, Fantasy"
939885,1297763,Batman Ninja vs. Yakuza League,5.7,22.0,2025-03-17,tt32508210,18.282,"Animation, Action"
1059269,1438267,Gamad Machmad 4,4.4,8.0,2025-03-16,NaN,10.171,"Drama, Comedy, Thriller, Action"
930641,1286773,The Metropolitan Opera: Fidelio,7.4,7.0,2025-03-15,NaN,11.995,Music
1042445,1417677,Bill Burr: Drop Dead Years,7.9,15.0,2025-03-14,tt34686029,6.094,Comedy
...,...,...,...,...,...,...,...,...
141908,248595,Alien Outlaw,2.7,13.0,1985-01-01,tt0190229,3.308,"Mystery, Science Fiction, Comedy, Western, Horror"
59004,82942,'Master Harold'... and the Boys,6.2,6.0,1985-01-01,tt0089564,4.057,Drama
127434,220343,Anna Karenina,5.2,6.0,1985-01-01,tt0088726,4.618,"Drama, Romance, TV Movie"
98676,161426,You Killed Me First,4.8,19.0,1985-01-01,tt0190157,0.671,"Drama, Horror"


In [19]:
films_df.head()

,id,title,vote_average,vote_count,release_date,imdb_id,popularity,genres
0,2,Ariel,7.1,340.0,1988-10-21,tt0094675,9.400,"Comedy, Drama, Romance, Crime"
1,3,Shadows in Paradise,7.3,403.0,1986-10-17,tt0092149,7.088,"Comedy, Drama, Romance"
2,5,Four Rooms,5.9,2678.0,1995-12-09,tt0113101,3.547,"Comedy, Crime"
3,6,Judgment Night,6.5,333.0,1993-10-15,tt0107286,12.110,"Action, Crime, Thriller"
4,8,Life in Loops (A Megacities RMX),7.5,27.0,2006-01-01,tt0825671,3.203,Documentary


In [20]:
genres_films = films_df.copy()

In [21]:
# Data preprocessing
# # One-hot encode the genres
ohe_films_df, genre_list_mlb = pre.one_hot_encode_genres(genres_films)

In [22]:
ohe_films_df.columns

Index(['id', 'title', 'vote_average', 'vote_count', 'release_date', 'imdb_id',
       'popularity', 'Action', 'Adventure', 'Animation', 'Comedy', 'Crime',
       'Documentary', 'Drama', 'Family', 'Fantasy', 'History', 'Horror',
       'Music', 'Mystery', 'Romance', 'Science Fiction', 'TV Movie',
       'Thriller', 'War', 'Western'],
      dtype='object')

In [23]:
if load_original_credits:
    # Gather info on directors and cast
    credits_df = pre.condense_credits(credits_df)
    data_loader.save_credits(credits_df)
# The credits DataFrame already has the key info in it if not. 

In [45]:
# importlib.reload(data_loader)
# data_loader.save_credits(credits_df)

In [24]:
print(type(ohe_films_df), type(credits_df))  # Debugging line

<class 'pandas.core.frame.DataFrame'> <class 'pandas.core.frame.DataFrame'>


In [25]:
# Tidy the noise and merge the credits' metadata with the films DataFrame
films_df = pre.data_tidying(ohe_films_df, credits_df)
# print(films_df.columns)


In [26]:
# Generate the user profile
user_ratings_df = user_pf.load_user_ratings()
user_profile = user_pf.create_user_profile(USER_ID, films_df, user_ratings_df, genre_list_mlb)

In [27]:
# Append the user profile to the ratings DataFrame
ratings_df = pd.concat([ratings_df, user_ratings_df], ignore_index=True)

In [28]:
# print all the values for vote_average in films_df
print((films_df['vote_count'] > 100).sum())

11368


In [29]:
films_df[films_df['vote_average'] != 0]

,id,title,vote_average,vote_count,release_date,imdb_id,popularity,Action,Adventure,Animation,...,Music,Mystery,Romance,Science Fiction,TV Movie,Thriller,War,Western,cast_info,director_info
0,2,Ariel,7.100,340.0,1988-10-21,tt0094675,9.400,0,0,0,...,0,0,1,0,0,0,0,0,"[(Turo Pajala, 54768), (Susanna Haavisto, 5476...","(Aki Kaurismäki, 2)"
1,3,Shadows in Paradise,7.300,403.0,1986-10-17,tt0092149,7.088,0,0,0,...,0,0,1,0,0,0,0,0,"[(Matti Pellonpää, 4826), (Kati Outinen, 5999)...","(Aki Kaurismäki, 3)"
2,5,Four Rooms,5.900,2678.0,1995-12-09,tt0113101,3.547,0,0,0,...,0,0,0,0,0,0,0,0,"[(Tim Roth, 3129), (Antonio Banderas, 3131), (...","(Allison Anders, 5)"
3,6,Judgment Night,6.500,333.0,1993-10-15,tt0107286,12.110,1,0,0,...,0,0,0,0,0,1,0,0,"[(Emilio Estevez, 2880), (Cuba Gooding Jr., 97...","(Stephen Hopkins, 6)"
4,12,Finding Nemo,7.800,19500.0,2003-05-30,tt0266543,8.524,0,0,1,...,0,0,0,0,0,0,0,0,"[(Albert Brooks, 13), (Ellen DeGeneres, 14), (...","(Andrew Stanton, 12)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28828,463800,Firebase,6.800,145.0,2017-06-28,tt7078926,6.090,1,0,0,...,0,0,0,1,0,0,1,0,"[(Steve Boyle, 190921), (Nic Rhind, 964251), (...","(Neill Blomkamp, 463800)"
28829,463906,The Saint,5.300,280.0,2017-07-11,tt2569088,9.959,1,1,0,...,0,0,0,0,0,0,0,0,"[(Adam Rayner, 144292), (Eliza Dushku, 13446),...","(Ernie Barbarash, 463906)"
28830,464111,Zygote,7.000,167.0,2017-07-12,tt7078780,7.634,0,0,0,...,0,0,0,1,0,0,0,0,"[(Dakota Fanning, 501), (Jose Pablo Cantillo, ...","(Neill Blomkamp, 464111)"
28831,464207,The Truth Is in the Stars,7.067,15.0,2017-05-01,tt7104950,3.632,0,0,0,...,0,0,0,0,0,0,0,0,"[(William Shatner, 1748), (Neil deGrasse Tyson...","(Craig Thompson, 464207)"


In [39]:
import hybrid_recommender as hyb
importlib.reload(hyb)

<module 'hybrid_recommender' from '/Users/Cathal/Rec-Genie/rec-sys/hybrid_recommender.py'>

In [40]:
# Generate recommendations
recommendations = hyb.hybrid_recommend(user_profile, films_df, credits_df, ratings_df, genre_list_mlb)

999999
<class 'int'>
User-User algorithm set up!


In [34]:
len(recommendations)

400

In [41]:
score_breakdown = hyb.score_breakdown(films_df, recommendations)

In [42]:
score_breakdown

,id,title,release_date,vote_average,vote_count,score,cast_score,director_score,genre_score,user_user_score
143,272,Batman Begins,2005-06-10,7.700,21297.0,12.571658,5.40,4.88,7.4620,3.286298
775,1894,Star Wars: Episode II - Attack of the Clones,2002-05-15,6.600,13441.0,12.434603,7.98,2.92,5.9840,3.129083
569,1420,Breakfast on Pluto,2005-11-16,7.200,396.0,9.970016,4.08,2.00,7.4190,3.636696
1045,2567,The Aviator,2004-12-17,7.221,5405.0,9.663816,3.63,1.80,9.8580,3.102576
84,155,The Dark Knight,2008-07-16,8.519,33561.0,9.320366,1.32,4.88,6.3795,3.194106
...,...,...,...,...,...,...,...,...,...,...
1027,2444,The Red Squirrel,1993-04-21,6.500,68.0,5.933704,0.00,0.00,7.4190,3.856384
18761,105130,Dangan Runner,1996-11-09,6.800,23.0,5.930737,0.00,0.00,5.8360,4.296657
14693,59065,Route Irish,2011-03-16,6.100,88.0,5.928122,0.00,0.00,9.8580,3.167882
22422,210577,Gone Girl,2014-10-01,7.890,18883.0,5.927634,0.75,1.68,4.4700,2.975034
